# Практика №1. Дерево решений

**Задача:** классифицировать автомобиль по шести категориальным характеристикам (цена покупки, стоимость обслуживания, число дверей, вместимость, размер багажника и безопасность).

**Датасет:** UCI Car Evaluation, 1 728 наблюдений, 6 категориальных признаков и 4 класса (`unacc`, `acc`, `good`, `vgood`). Набор также приводится в материалах лекции. Источник: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/19/car-evaluation).

Дерево решений строит последовательность правил и не требует масштабирования признаков. Для категориальных столбцов применим One-Hot Encoding.

In [ ]:
%pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 1. Загрузка данных

Получаем признаки и класс автомобиля из UCI. Проверяем размер таблицы, имена столбцов и распределение целевой переменной.

In [ ]:
dataset = fetch_ucirepo(id=19)
X_raw = dataset.data.features.copy()
y_raw = dataset.data.targets.iloc[:, 0].copy()

print(f"Размер матрицы признаков: {X_raw.shape}")
print("Признаки:", X_raw.columns.tolist())
print("Целевая переменная:", dataset.data.targets.columns[0])
display(X_raw.head())
display(pd.Series(y_raw).value_counts().to_frame("count"))

## 2. Проверка качества и очистка

Ищем пропуски, специальные маркеры отсутствующих значений, дубликаты и значения категорий, не предусмотренные описанием UCI. Повторяющиеся строки удаляются. Оставшиеся пропуски, если они есть, будут заполнены наиболее частой категорией внутри Pipeline.

In [ ]:
df = X_raw.copy()
df["target"] = y_raw.astype("object")
df = df.replace("?", np.nan)

print("Пропуски до очистки:")
display(df.isna().sum().to_frame("missing"))
print("Количество полных дубликатов:", int(df.duplicated().sum()))

allowed_values = {
    "buying": {"vhigh", "high", "med", "low"},
    "maint": {"vhigh", "high", "med", "low"},
    "doors": {"2", "3", "4", "5more"},
    "persons": {"2", "4", "more"},
    "lug_boot": {"small", "med", "big"},
    "safety": {"low", "med", "high"},
}
for column, allowed in allowed_values.items():
    if column in df.columns:
        observed = set(df[column].dropna().astype(str).unique())
        print(f"Недопустимые значения в {column}: {sorted(observed - allowed)}")

rows_before = len(df)
df = df.dropna(subset=["target"]).drop_duplicates().reset_index(drop=True)
print(f"Удалено строк с отсутствующей целью/дубликатов: {rows_before - len(df)}")
print("Пропуски в признаках после проверки:")
display(df.drop(columns="target").isna().sum().to_frame("missing"))

X = df.drop(columns="target").astype("object")
y = df["target"].astype(str)
print("Частоты редких категорий (для дополнительного контроля):")
for column in X.columns:
    print(column, X[column].value_counts(dropna=False).tail(3).to_dict())

## 3. Разведочный анализ данных (EDA)

Построим распределения значений по каждому категориальному признаку и распределение классов. Для корреляционной матрицы преобразуем категории в бинарные индикаторы методом One-Hot Encoding. Корреляции бинарных индикаторов используются только как разведочный инструмент.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for column, ax in zip(X.columns, axes.ravel()):
    counts = X[column].value_counts()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(f"Распределение: {column}")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
class_counts = y.value_counts()
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Распределение целевых классов")
plt.xlabel("Класс оценки автомобиля")
plt.tight_layout()
plt.show()

encoded_eda = pd.get_dummies(X, dummy_na=True, dtype=int)
plt.figure(figsize=(14, 11))
sns.heatmap(encoded_eda.corr(), cmap="coolwarm", center=0, xticklabels=True, yticklabels=True)
plt.title("Корреляционная матрица One-Hot признаков")
plt.tight_layout()
plt.show()

## 4. Разделение выборки и подготовка категорий

Применяем стратификацию, чтобы соотношение четырёх классов оставалось сопоставимым во всех частях выборки. Pipeline сначала заполнит пропуски модой, затем закодирует категории. Отдельное масштабирование не требуется для дерева решений.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)
print(f"Train: {len(X_train)}, validation: {len(X_val)}, test: {len(X_test)}")

categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

## 5. Подбор сложности дерева на validation

Глубокое дерево может переобучиться. Сравним несколько значений `max_depth` и `min_samples_leaf`, выбирая конфигурацию по Macro F1, которая учитывает качество распознавания всех классов.

In [ ]:
validation_results = []
best_params = None
best_val_f1 = -1.0

for max_depth in [4, 8, None]:
    for min_samples_leaf in [1, 3, 5]:
        candidate = Pipeline([
            ("preprocessor", categorical_preprocessor),
            ("model", DecisionTreeClassifier(
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ])
        candidate.fit(X_train, y_train)
        prediction = candidate.predict(X_val)
        score = f1_score(y_val, prediction, average="macro")
        validation_results.append({
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "validation_macro_f1": score,
        })
        if score > best_val_f1:
            best_val_f1 = score
            best_params = {"max_depth": max_depth, "min_samples_leaf": min_samples_leaf}

validation_table = pd.DataFrame(validation_results).sort_values("validation_macro_f1", ascending=False)
display(validation_table)
print("Лучшие параметры:", best_params)

## 6. Финальное обучение и оценка на test

Повторно обучаем модель на train + validation и рассчитываем метрики только на ранее отложенной тестовой части. Используем Accuracy, Balanced Accuracy и Macro F1, так как классы представлены неравномерно.

In [ ]:
X_train_full = pd.concat([X_train, X_val], axis=0)
y_train_full = pd.concat([y_train, y_val], axis=0)

final_model = Pipeline([
    ("preprocessor", categorical_preprocessor),
    ("model", DecisionTreeClassifier(
        **best_params,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])
final_model.fit(X_train_full, y_train_full)
y_test_pred = final_model.predict(X_test)

test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_test_pred),
    "Macro F1": f1_score(y_test, y_test_pred, average="macro"),
    "Weighted F1": f1_score(y_test, y_test_pred, average="weighted"),
}
display(pd.Series(test_metrics).to_frame("value"))
print(classification_report(y_test, y_test_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, cmap="Blues", xticks_rotation=30)
plt.title("Матрица ошибок: Decision Tree")
plt.tight_layout()
plt.show()

## 7. Интерпретация дерева и сохранение модели

Дерево можно визуализировать в сокращённом виде (первые три уровня), чтобы показать принцип последовательного разбиения признаков. Сохраняем Pipeline целиком, включая кодирование категорий.

In [ ]:
fitted_preprocessor = final_model.named_steps["preprocessor"]
encoded_feature_names = fitted_preprocessor.named_steps["onehot"].get_feature_names_out(X.columns)
tree_model = final_model.named_steps["model"]

plt.figure(figsize=(22, 10))
plot_tree(
    tree_model,
    max_depth=3,
    feature_names=encoded_feature_names,
    class_names=[str(value) for value in tree_model.classes_],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Дерево решений (показаны первые уровни)")
plt.tight_layout()
plt.show()

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, artifact_dir / "decision_tree_car_evaluation.joblib")
with (artifact_dir / "decision_tree_metrics.json").open("w", encoding="utf-8") as file:
    json.dump({"best_params": best_params, "validation_macro_f1": best_val_f1, "test": test_metrics}, file, ensure_ascii=False, indent=2)
print("Модель и метрики сохранены в папку artifacts/")

## 8. Вывод

В заключении сравните качество распознавания отдельных классов и сделайте вывод о влиянии глубины дерева на обобщающую способность. При интерпретации учитывайте, что визуализация показывает только верхние уровни полного дерева.